# Leksjon 1: Python Tenker Som en Hacker

**Velkommen!** I denne leksjonen skal vi lære det grunnleggende i Python-programmering ved å bruke det som et cybersikkerhetsverktøy. I stedet for bare å skrive "Hello World", skal vi lære hvordan hackere og sikkerhetseksperter ser på nettet.

## 1. Introduksjon: Angriperens vs. Forsvarerens Tankesett
Cybersikkerhet handler ikke bare om verktøy; det handler om et tankesett. 
- **Angripere** leter etter **ett** svakt punkt for å komme seg inn.
- **Forsvarere** må sikre **alle** punkter.

**Etikk:** Ferdighetene du lærer her er kraftige. Uautorisert testing av systemer du ikke eier er ulovlig og uetisk. Ha alltid tillatelse før du skanner eller tester et nettsted. Vi bruker trygge, pedagogiske mål i dag.

## 2. Python Grunnleggende: Variabler og Tekst
I Python lagrer vi data i **variabler**. Tekst kalles en **String** (streng).

La oss se på et enkelt eksempel der vi lagrer en URL.

In [3]:
# Lage en variabel
target_url = "https://example.com"

# Skrive ut variabelen
print("Mål:", target_url)

Mål: https://example.com


### Lister og Løkker
Ofte vil vi jobbe med flere ting samtidig (som en liste med passord eller URL-er). Vi bruker **Lister** for lagring og **Løkker** (Loops) for å gå gjennom dem.

In [4]:
# En liste over vanlige admin-stier
admin_paths = ["/admin", "/login", "/dashboard", "/config"]

# Gå gjennom listen med en løkke
for path in admin_paths:
    full_url = target_url + path
    print("Sjekker potensiell sti:", full_url)

Sjekker potensiell sti: https://example.com/admin
Sjekker potensiell sti: https://example.com/login
Sjekker potensiell sti: https://example.com/dashboard
Sjekker potensiell sti: https://example.com/config


## 3. `requests`-biblioteket
Nettlesere som Chrome eller Firefox sender **HTTP-forespørsler** til servere for å hente nettsider. Python kan også gjøre dette ved hjelp av `requests`-biblioteket. Slik samhandler skript med nettet.

In [5]:
import requests
import urllib3

# Undertrykk advarselen om at vi gjør en ubekreftet HTTPS-forespørsel
# I produksjon ville du fikset sertifikatproblemet i stedet for å undertrykke det.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Send en GET-forespørsel til målet
# verify=False brukes ofte i sikkerhetstesting for å omgå SSL-feil fra proxyer eller selvsignerte sertifikater
response = requests.get(target_url, verify=False)

# Sjekk statuskoden (200 betyr OK)
print("Statuskode:", response.status_code)

# Begrens utskriften til de første 500 tegnene så vi ikke spammer skjermen
print("Forhåndsvisning av sideinnhold:\n", response.text[:500])

Statuskode: 200
Forhåndsvisning av sideinnhold:
 <!doctype html><html lang="en"><head><title>Example Domain</title><meta name="viewport" content="width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-family:system-ui,sans-serif}h1{font-size:1.5em}div{opacity:0.8}a:link,a:visited{color:#348}</style></head><body><div><h1>Example Domain</h1><p>This domain is for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.org/domains/example">Learn more<


## 4. Lese HTTP-overskrifter (Headers)
Når en server svarer, sender den **Headers** (metadata) før selve HTML-innholdet. Disse overskriftene kan fortelle oss mye om sikkerheten til serveren.

Vanlige sikkerhetsoverskrifter vi ser etter:
- `X-Frame-Options`: Hindrer Clickjacking.
- `Strict-Transport-Security` (HSTS): Tvinger HTTPS.
- `Content-Security-Policy` (CSP): Hindrer XSS.

La oss inspisere overskriftene til målet vårt.

In [6]:
# Få tilgang til overskrifter (headers) fra responsobjektet
headers = response.headers

# Skriv ut alle overskrifter pent
for key, value in headers.items():
    print(f"{key}: {value}")

print("-" * 30)

# Sjekk etter en spesifikk sikkerhetsoverskrift
security_header = "X-Frame-Options"

if security_header in headers:
    print(f"[+] Fant {security_header}: {headers[security_header]}")
else:
    print(f"[-] Mangler {security_header} - Potensiell sårbarhet!")

Date: Wed, 25 Mar 2026 10:04:49 GMT
Content-Type: text/html
Transfer-Encoding: chunked
Connection: keep-alive
Content-Encoding: gzip
last-modified: Tue, 24 Mar 2026 22:07:32 GMT
allow: GET, HEAD
Age: 1991
cf-cache-status: HIT
Vary: Accept-Encoding
Server: cloudflare
CF-RAY: 9e1d17f8587da8b6-CPH
------------------------------
[-] Mangler X-Frame-Options - Potensiell sårbarhet!


## 5. Introduksjon til XSS (Cross-Site Scripting)

**XSS** oppstår når en angriper kan injisere ondsinnet kode (vanligvis JavaScript) inn i et nettsted som andre brukere ser.

Det mest grunnleggende tegnet på XSS-sårbarhet er når en side reflekterer brukerinndata uten å rense det ("sanitize"). Vi ser ofte etter `<script>`-taggen som et bevis på konseptet (Proof of Concept).

La oss prøve å finne en `<script>`-tag i en bit med HTML-kode.

In [7]:
# Simulert HTML-innhold (tenk deg at dette kom fra et nettsted)
html_content = """
<html>
  <body>
    <h1>Velkommen!</h1>
    <p>Brukerkommentar: <script>alert('Hacked!');</script></p>
  </body>
</html>
"""

# Manuelt søk med Python
if "<script>" in html_content:
    print("FARE: Mulig XSS-vektor funnet! <script>-tag oppdaget.")
else:
    print("Trygt: Ingen <script>-tagger funnet.")

FARE: Mulig XSS-vektor funnet! <script>-tag oppdaget.


### Din Tur: Automatiser Skanneren

Kombiner det vi har lært! Skriv et skript nedenfor som:
1. Henter en URL (spør brukeren om inndata eller bruk en variabel).
2. Sjekker om `Content-Security-Policy` mangler.
3. Sjekker om teksten "\<script>" vises i innholdet (forenklet sjekk).

*(Merk: Virkelige skannere er mye mer komplekse, men dette er logikken!)*

In [ ]:
# Skriv koden din her


## Oppsummering

I dag lærte vi:
1. **Variabler og Løkker** lar oss automatisere repetitive oppgaver.
2. **Requests** lar Python snakke med nettet.
3. **Headers** avslører sikkerhetskonfigurasjoner.
4. Små tekstmønstre (som `<script>`) kan indikere store sårbarheter.

**Neste Leksjon:** Vi skal dykke ned i filer, logger og digital etterforskning (forensics)!

In [ ]:
# Løsning for "Din Tur"-oppgaven
# Fjern kommentaren (#) for å se løsningen

# url = input("Skriv inn URL å skanne: ")
# try:
#     r = requests.get(url)
#     if "Content-Security-Policy" not in r.headers:
#         print("[-] Mangler Content-Security-Policy header")
#     if "<script>" in r.text:
#         print("[!] <script>-tag funnet i kroppen (krever manuell verifisering)")
# except Exception as e:
#     print("Feil ved henting av URL:", e)